### What is Incremental File Ingestion?

Incremental file ingestion means processing only new or changed files instead of processing all files again and again.

### Process all files from folder

In [15]:
import glob
import pandas as pd

#glob is used to search files using patterns

file_path = "E:/data/Documents/Data_Engineering/Module1_Data_Source/module1-dataset/incremental_practice_file_ingestion/"
files = glob.glob(file_path+"*.csv") 

all_data = []

for file_path in files:
    df = pd.read_csv(file_path)
    all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True) #ignore_index=True, pandas creates a fresh index

combined_df.head()

,order_id,customer_id,order_date,amount,status
0,1001,1035.0,2026-04-01,4743,Cancelled
1,1002,1030.0,2026-04-01,1713,Pending
2,1003,1005.0,2026-04-01,1803,Completed
3,1004,1035.0,2026-04-01,4206,Completed
4,1005,1041.0,2026-04-01,4604,Cancelled


### Incremental pipeline

In [9]:
import os
from datetime import datetime

def get_file_metadata(file_path):
    file_name = os.path.basename(file_path)
    file_size = os.path.getsize(file_path)

    modified_timestamp = os.path.getmtime(file_path)
    modified_time = datetime.fromtimestamp(modified_timestamp)

    return {
        "file_name": file_name,
        "file_size": file_size,
        "modified_time": modified_time
    }

In [11]:
def is_file_already_processed(file_metadata, log_file):
    
    if not os.path.exists(log_file): # Will check if log_file exists or not
        return False

    log_df = pd.read_csv(log_file)

    matched = log_df[
        (log_df["file_name"] == file_metadata["file_name"]) &
        (log_df["file_size"] == file_metadata["file_size"]) &
        (log_df["modified_time"] == file_metadata["modified_time"]) &
        (log_df["status"] == "SUCCESS")
    ]

    return not matched.empty

In [3]:

import glob
import os
import pandas as pd
from datetime import datetime

path = "E:/data/Documents/Data_Engineering/Module1_Data_Source/module1-dataset/incremental_practice_file_ingestion/"
log_file = "file_processing_log.csv"
output_file = "combined_output.csv"


full_path_log_file = path + log_file


# Step 1: Read old log file
if os.path.exists(full_path_log_file):
    log_df = pd.read_csv(full_path_log_file)
else:
    log_df = pd.DataFrame(columns=["file_name", "modified_time", "processed_time", "status"])


# Step 2: Get all CSV files from input folder
all_files = glob.glob(path + "*.csv")


# Step 3: Check each file
print("CHECKING FILE ONE BY ONE")
for file_path in all_files:
    file_name = os.path.basename(file_path)
    print("CHECKING FILE:", file_name)

    modified_time = datetime.fromtimestamp(
        os.path.getmtime(file_path)
    ).strftime("%Y-%m-%d %H:%M:%S")

    print("CHECKING FILE TIME:",modified_time)
    already_processed = False

    # Compare current file with log file
    for index, row in log_df.iterrows(): 
        if (
            row["file_name"] == file_name
            and row["modified_time"] == modified_time
            and row["status"] == "SUCCESS"
        ):
            already_processed = True
            break

    if already_processed:
        print("Skipping already processed file", file_name)
        continue

    print("Processing file:", file_name)

    # Read current CSV file
    df = pd.read_csv(file_path)

    # Add tracking columns
    df["source_file"] = file_name
    df["file_modified_time"] = modified_time
    df["ingestion_time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Write data to combined output file
    full_path_output_file = path + output_file
    print("OUTPUT FILE LOCATION:  ",full_path_output_file)
    if os.path.exists(full_path_output_file):
        df.to_csv(full_path_output_file, mode="a", index=False, header=False) # APPEND MODE
    else:
        df.to_csv(full_path_output_file, mode="w", index=False, header=True) # WRITE MODE

    # Create new log entry
    new_log = pd.DataFrame({
        "file_name": [file_name],
        "modified_time": [modified_time],
        "processed_time": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
        "status": ["SUCCESS"]
    })

    # Append new log entry
    print("LOG FILE LOCATION:  ", full_path_log_file)
    if os.path.exists(full_path_log_file):
        new_log.to_csv(full_path_log_file, mode="a", index=False, header=False)
    else:
        new_log.to_csv(full_path_log_file, mode="w", index=False, header=True)


CHECKING FILE ONE BY ONE
CHECKING FILE: orders_2026_04_01.csv
CHECKING FILE TIME: 2026-04-30 23:12:33
Processing file: orders_2026_04_01.csv
OUTPUT FILE LOCATION:   E:/data/Documents/Data_Engineering/Module1_Data_Source/module1-dataset/incremental_practice_file_ingestion/combined_output.csv
LOG FILE LOCATION:   E:/data/Documents/Data_Engineering/Module1_Data_Source/module1-dataset/incremental_practice_file_ingestion/file_processing_log.csv
CHECKING FILE: orders_2026_04_02.csv
CHECKING FILE TIME: 2026-04-30 23:12:33
Processing file: orders_2026_04_02.csv
OUTPUT FILE LOCATION:   E:/data/Documents/Data_Engineering/Module1_Data_Source/module1-dataset/incremental_practice_file_ingestion/combined_output.csv
LOG FILE LOCATION:   E:/data/Documents/Data_Engineering/Module1_Data_Source/module1-dataset/incremental_practice_file_ingestion/file_processing_log.csv
CHECKING FILE: orders_2026_04_03.csv
CHECKING FILE TIME: 2026-04-30 23:12:33
Processing file: orders_2026_04_03.csv
OUTPUT FILE LOCATION: